# Colab smoke — Duckietown + SB3 en Python 3.11

Receta reproducible para entrenar/evaluar en **Duckietown real** sobre Colab.
Equivalente ejecutable de `COLAB_SETUP.md`. El kernel de Colab es 3.12; montamos
un **venv 3.11** y ejecutamos todo como subprocess con `PY`.

Orden de pruebas (menor a mayor riesgo): imports → smoke mock → reset real →
wrappers/shapes → PPO corto → eval.

## 1. Clonar el repositorio (privado)

In [ ]:
from getpass import getpass
import os
os.environ['GH_PAT'] = getpass('GitHub PAT: ')
!git clone https://$GH_PAT@github.com/AdolfoPM02/MAML.git
%cd MAML

## 2. Python 3.11 + venv + dependencias de sistema

In [ ]:
!sudo add-apt-repository -y ppa:deadsnakes/ppa && sudo apt-get update -qq
!sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev xvfb python3-opengl ffmpeg freeglut3-dev
!python3.11 -m venv /content/venv-maml
PY = '/content/venv-maml/bin/python'
!{PY} -m pip install -U pip wheel setuptools
!{PY} --version

## 3. Instalar stack ML + Duckietown real (sin romper numpy)
numpy `1.26.4` es el puente que satisface a SB3/torch/gymnasium y a Duckietown.

In [ ]:
!{PY} -m pip install "numpy==1.26.4" "stable-baselines3==2.8.0" torch "opencv-python" "gymnasium==1.2.3" pyvirtualdisplay
!{PY} -m pip install --no-deps "git+https://github.com/duckietown/gym-duckietown.git@daffy"
!{PY} -m pip install "pyglet==1.5.27" pyzmq PyYAML scikit-image pillow
!{PY} -m pip install "numpy==1.26.4" --force-reinstall --no-deps

## 4. Verificar imports

In [ ]:
!{PY} -c "import numpy,torch,cv2,gymnasium,stable_baselines3; import gym_duckietown; print('numpy',numpy.__version__,'torch',torch.__version__); print('duckietown OK')"

## 5. Smoke tests con MOCK (sin display)

In [ ]:
!{PY} scripts/smoke_test_phase2.py
!{PY} scripts/smoke_test_model_load.py

## 6. Duckietown REAL — reset()

In [ ]:
!xvfb-run -a -s "-screen 0 1024x768x24" {PY} scripts/check_duckie_real.py --reset-only

## 7. Wrappers + shapes con Duckietown REAL

In [ ]:
!xvfb-run -a {PY} scripts/check_duckie_real.py

## 8. Entrenar un PPO corto real

In [ ]:
!xvfb-run -a {PY} train.py --algo ppo --map Duckietown-loop_empty-v0 --timesteps 5000 --output ppo_colab_test

## 9. Evaluar el modelo real (incluye prueba del mapa oculto)

In [ ]:
!xvfb-run -a {PY} eval.py --algo ppo --model models/ppo_colab_test --map Duckietown-loop_empty-v0 --episodes 3
!xvfb-run -a {PY} eval.py --algo ppo --model models/ppo_colab_test --map Duckietown-loop_obstacles-v0 --episodes 3 --allow-eval

## 10. Preparar la entrega (cuando el entrenamiento completo esté listo)
**Aún no**: requiere el entrenamiento real completo. Ver `COLAB_SETUP.md` §10.
```
!cp models/<mejor_modelo>.zip models/best_duckie_agent.zip
!{PY} -m pip freeze > requirements.txt
```
Luego hacer el dry-run del contrato en un Colab nuevo.